# GraviNum - N-dimensional arrays and linear algebra

Build the solution first so the notebook can reference the assemblies:

```
dotnet build Gravicode.Science.sln -c Release
```

*Dibuat oleh Gravicode Studios, dipimpin oleh Kang Fadhil*

In [ ]:
#r "../src/GraviNum/bin/Release/net10.0/Gravicode.Science.GraviNum.dll"
#r "nuget: ScottPlot, 5.1.59"

using Gravicode.Science.GraviNum;
using Gravicode.Science.GraviNum.Compute;

Console.WriteLine(GraviInfo.HardwareReport());

## Creating arrays

Every array is a shared buffer plus a shape, strides and an offset, so reshaping and transposing cost nothing.

In [ ]:
var a = NdArray.Arange(12).Reshape(3, 4);
Console.WriteLine(a);

Console.WriteLine($"transpose shares the buffer: a[1,2]={a[1, 2]}, a.T[2,1]={a.T[2, 1]}");
Console.WriteLine($"a[:, 1:3] is a strided view, contiguous = {a.Slice(Slice.All, Slice.Range(1, 3)).IsContiguous}");

## Broadcasting

A length-4 vector stretches across all three rows without being copied.

In [ ]:
var matrix = NdArray.Ones(3, 4);
var offsets = NdArray.Arange(4);
Console.WriteLine(matrix + offsets);

## Linear algebra

In [ ]:
var m = NdArray.FromArray(new double[,] { { 4, 7, 2 }, { 3, 6, 1 }, { 2, 5, 9 } });

Console.WriteLine($"det  = {LinAlg.Determinant(m):F4}");
Console.WriteLine($"rank = {LinAlg.MatrixRank(m)}");
Console.WriteLine($"cond = {LinAlg.ConditionNumber(m):F4}");

var svd = Decomposition.Svd(m);
Console.WriteLine($"singular values: {string.Join(", ", svd.SingularValues.ToArray().Select(v => v.ToString("F4")))}");
Console.WriteLine($"U S V' reconstructs A: {UFunc.AllClose(svd.Reconstruct(), m, 1e-9)}");

## Statistics

In [ ]:
var rng = new GraviRandom(42);
var samples = rng.Normal(100, 15, 100_000);

foreach (var (key, value) in Statistics.Describe(samples))
    Console.WriteLine($"{key,-8}{value,12:F4}");

## Heatmap

ScottPlot renders directly into the notebook.

In [ ]:
var heat = NdArray.Zeros(40, 40);
for (var i = 0; i < 40; i++)
    for (var j = 0; j < 40; j++)
    {
        var u = (i - 20) / 8.0;
        var v = (j - 20) / 8.0;
        heat[i, j] = Math.Exp(-(u * u + v * v) / 2) * Math.Cos(u * 2) * Math.Sin(v * 2);
    }

var plot = new ScottPlot.Plot();
plot.Add.Heatmap(heat.To2DArray());
plot.Title("GraviNum - matrix heatmap");
plot.GetImageHtml(800, 600)

## CPU versus GPU

The GPU only wins once the matrix is big enough to hide the transfer cost.

In [ ]:
using System.Diagnostics;

foreach (var size in new[] { 128, 256, 512, 1024 })
{
    var left = rng.StandardNormal(size, size);
    var right = rng.StandardNormal(size, size);

    var watch = Stopwatch.StartNew();
    LinAlg.Dot(left, right);
    watch.Stop();

    var gflops = 2.0 * size * size * size / watch.Elapsed.TotalSeconds / 1e9;
    Console.WriteLine($"{size,5} x {size,-5}{watch.ElapsedMilliseconds,7} ms{gflops,9:F2} GFLOP/s");
}

Console.WriteLine(Compute.DescribeDevices());